# FastConformer CTC Vietnamese Fine-Tune Main Notebook

<style>
.jp-RenderedMarkdown, .jp-MarkdownOutput, .rendered_html, .text_cell_render, .markdown-cell {
  font-family: "Noto Sans", "Segoe UI", "Arial", "DejaVu Sans", sans-serif !important;
}
.jp-RenderedMarkdown code, .jp-MarkdownOutput code, .rendered_html code, .text_cell_render code {
  font-family: "JetBrains Mono", "Consolas", "DejaVu Sans Mono", monospace !important;
}
</style>

Notebook chính đã được tách cell để dễ chạy/debug trên Kaggle.

Run order: chạy lần lượt từ trên xuống. Các phase được chạy ở từng cell riêng: Phase 1, Phase 2, Phase 3, rồi Finalize.


In [16]:
# Kaggle setup: run once if dependencies are missing.
!pip install -q "nemo_toolkit[asr]" huggingface_hub jiwer sentencepiece soundfile pandas


## 1. Imports and Configuration


In [17]:
"""
Kaggle-ready controlled fine-tuning pipeline for
`nvidia/stt_en_fastconformer_ctc_large` on Vietnamese VIVOS.

Run on Kaggle after installing dependencies, for example:

    pip install -q "nemo_toolkit[asr]" huggingface_hub jiwer sentencepiece soundfile pandas
    python finetune_fastconformer_vi_best_checkpoint.py

The pipeline follows three phases:
1. Decoder warmup with frozen encoder.
2. Partial encoder adaptation with the last encoder blocks unfrozen.
3. Full fine-tune with a low learning rate.

Each phase keeps only the best checkpoint by `val_wer`, keeps `last.ckpt`
for resume, and exports the best `.nemo` model for the next phase.
"""

from __future__ import annotations

import gc
import gzip
import json
import math
import os
import random
import re
import shutil
import tarfile
import time
import unicodedata
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import pandas as pd
import sentencepiece as spm
import soundfile as sf
import torch
from huggingface_hub import snapshot_download
from jiwer import cer, wer
from omegaconf import OmegaConf

import nemo.collections.asr as nemo_asr

try:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import Callback, EarlyStopping, ModelCheckpoint
except ImportError:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import Callback, EarlyStopping, ModelCheckpoint


SEED = 42
random.seed(SEED)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("NEMO_LOG_LEVEL", "INFO")

ROOT = Path("/kaggle/working/vivos")
EXTRACT_DIR = ROOT / "data" / "extracted"
MANIFEST_DIR = Path("/kaggle/working/vivos_manifest_phase_best")
TOKENIZER_ROOT = Path("/kaggle/working/tokenizers")
TOKENIZER_DIR = TOKENIZER_ROOT / "tokenizer_spe_unigram_v1024_vi"
CORPUS_PATH = TOKENIZER_ROOT / "vivos_train_only_corpus_vi.txt"

MODEL_NAME = "nvidia/stt_en_fastconformer_ctc_large"
DATASET_ID = "AILAB-VNUHCM/vivos"

EXP_DIR = Path("/kaggle/working/nemo_experiments")
EXP_NAME = "fastconformer_vi"
PHASE_ROOT = EXP_DIR / EXP_NAME
ARTIFACT_DIR = Path("/kaggle/working/fastconformer_vi_artifacts")

PHASE1_BEST_NEMO = Path("/kaggle/working/phase1_best_decoder_only.nemo")
PHASE2_BEST_NEMO = Path("/kaggle/working/phase2_best_partial_encoder.nemo")
PHASE3_BEST_NEMO = Path("/kaggle/working/phase3_best_full_finetune.nemo")
OVERALL_BEST_NEMO = Path("/kaggle/working/fastconformer_vi_best_model.nemo")

TRAIN_MANIFEST = MANIFEST_DIR / "train_manifest.json"
VAL_MANIFEST = MANIFEST_DIR / "val_manifest.json"
TEST_MANIFEST = MANIFEST_DIR / "test_manifest.json"

EVAL_CSV = ARTIFACT_DIR / "phase_eval_summary.csv"
TRAINING_CSV = ARTIFACT_DIR / "phase_training_summary.csv"
SAMPLE_PREDICTIONS_CSV = ARTIFACT_DIR / "sample_predictions.csv"

TOKENIZER_VOCAB_SIZE = 1024
TOKENIZER_MODEL_TYPE = "unigram"
RETRAIN_TOKENIZER = True

MIN_DURATION = 0.1
MAX_DURATION = 8.0
EVAL_BATCH_SIZE = 8
NUM_SAMPLE_PREDICTIONS_PER_EPOCH = 10

PRECISION = "16-mixed"
GRADIENT_CLIP_VAL = 1.0
WEIGHT_DECAY = 0.001
WARMUP_STEPS = 500
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.0

# Set to True and point the phase entry at its `last.ckpt` to resume after a
# Kaggle interruption. Resume is exact only within the same phase.
RESUME_FROM_LAST = False
RESUME_CKPT_BY_PHASE = {
    "phase1_decoder_warmup": "",
    "phase2_partial_encoder": "",
    "phase3_full_finetune": "",
}


@dataclass(frozen=True)
class PhaseConfig:
    name: str
    output_nemo: Path
    init_nemo: Path | None
    trainable_mode: str
    max_epochs: int
    lr: float
    batch_size: int
    accumulate_grad_batches: int
    unfrozen_encoder_blocks: int = 0
    max_duration: float = MAX_DURATION


PHASES = [
    PhaseConfig(
        name="phase1_decoder_warmup",
        output_nemo=PHASE1_BEST_NEMO,
        init_nemo=None,
        trainable_mode="decoder_only",
        max_epochs=3,
        lr=5e-4,
        batch_size=4,
        accumulate_grad_batches=8,
    ),
    PhaseConfig(
        name="phase2_partial_encoder",
        output_nemo=PHASE2_BEST_NEMO,
        init_nemo=PHASE1_BEST_NEMO,
        trainable_mode="partial_encoder_last",
        max_epochs=10,
        lr=3e-5,
        batch_size=2,
        accumulate_grad_batches=16,
        unfrozen_encoder_blocks=6,
    ),
    PhaseConfig(
        name="phase3_full_finetune",
        output_nemo=PHASE3_BEST_NEMO,
        init_nemo=PHASE2_BEST_NEMO,
        trainable_mode="full_encoder_decoder",
        max_epochs=25,
        lr=3e-5,
        batch_size=2,
        accumulate_grad_batches=16,
    ),
]


VIETNAMESE_LETTER_RE = re.compile(
    r"[^0-9a-z"
    r"àáảãạăằắẳẵặâầấẩẫậ"
    r"èéẻẽẹêềếểễệ"
    r"ìíỉĩị"
    r"òóỏõọôồốổỗộơờớởỡợ"
    r"ùúủũụưừứửữự"
    r"ỳýỷỹỵđ"
    r"\s]"
)




## 2. Text Normalization


In [18]:
def normalize_transcript(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = VIETNAMESE_LETTER_RE.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_for_metric(text: str) -> str:
    return normalize_transcript(text)


## 3. Dataset and Manifest Preparation


In [19]:
def download_and_extract_vivos() -> None:
    archive_path = ROOT / "data" / "vivos.tar.gz"
    ROOT.mkdir(parents=True, exist_ok=True)
    if not archive_path.exists():
        snapshot_download(
            repo_id=DATASET_ID,
            repo_type="dataset",
            local_dir=str(ROOT),
            local_dir_use_symlinks=False,
        )

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    if not list(EXTRACT_DIR.rglob("*.wav")):
        with tarfile.open(archive_path, "r:gz") as tar:
            tar.extractall(EXTRACT_DIR)


def read_prompts(path: Path, wav_index: dict[str, str]) -> list[dict[str, Any]]:
    rows = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(" ", 1)
            if len(parts) != 2:
                continue
            utt_id, text = parts
            audio_path = wav_index.get(utt_id)
            if audio_path is None:
                continue
            info = sf.info(audio_path)
            rows.append(
                {
                    "utt_id": utt_id,
                    "audio_filepath": audio_path,
                    "duration": info.frames / info.samplerate,
                    "sample_rate": info.samplerate,
                    "text": normalize_transcript(text),
                }
            )
    return rows


def write_manifest(rows: list[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    {
                        "audio_filepath": row["audio_filepath"],
                        "duration": float(row["duration"]),
                        "text": row["text"],
                    },
                    ensure_ascii=False,
                )
                + "\n"
            )


def prepare_manifests() -> None:
    download_and_extract_vivos()
    wav_index = {path.stem: str(path) for path in EXTRACT_DIR.rglob("*.wav")}
    train_all = read_prompts(ROOT / "data" / "prompts-train.txt.gz", wav_index)
    test_rows = read_prompts(ROOT / "data" / "prompts-test.txt.gz", wav_index)

    random.Random(SEED).shuffle(train_all)
    val_size = max(1, int(len(train_all) * 0.05))
    val_rows = train_all[:val_size]
    train_rows = train_all[val_size:]

    write_manifest(train_rows, TRAIN_MANIFEST)
    write_manifest(val_rows, VAL_MANIFEST)
    write_manifest(test_rows, TEST_MANIFEST)

    audit_rows = []
    for split, rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
        durations = [row["duration"] for row in rows]
        sample_rates = Counter(row["sample_rate"] for row in rows)
        audit_rows.append(
            {
                "split": split,
                "rows": len(rows),
                "sample_rates": dict(sample_rates),
                "min_duration": min(durations),
                "max_duration": max(durations),
                "mean_duration": sum(durations) / len(durations),
                "filtered_by_max_duration_8s": sum(d > MAX_DURATION for d in durations),
            }
        )
    print(pd.DataFrame(audit_rows))


## 4. SentencePiece Tokenizer


In [20]:
def train_sentencepiece_tokenizer() -> None:
    if RETRAIN_TOKENIZER and TOKENIZER_DIR.exists():
        shutil.rmtree(TOKENIZER_DIR)

    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    TOKENIZER_ROOT.mkdir(parents=True, exist_ok=True)

    with open(CORPUS_PATH, "w", encoding="utf-8") as out:
        with open(TRAIN_MANIFEST, encoding="utf-8") as f:
            for line in f:
                out.write(json.loads(line)["text"] + "\n")

    model_file = TOKENIZER_DIR / "tokenizer.model"
    if not model_file.exists():
        spm.SentencePieceTrainer.Train(
            input=str(CORPUS_PATH),
            model_prefix=str(TOKENIZER_DIR / "tokenizer"),
            vocab_size=TOKENIZER_VOCAB_SIZE,
            model_type=TOKENIZER_MODEL_TYPE,
            character_coverage=1.0,
            unk_id=0,
            bos_id=-1,
            eos_id=-1,
            pad_id=-1,
            hard_vocab_limit=False,
            shuffle_input_sentence=True,
            input_sentence_size=1_000_000,
        )

    sp = spm.SentencePieceProcessor(model_file=str(model_file))
    vocab_txt = TOKENIZER_DIR / "vocab.txt"
    with open(vocab_txt, "w", encoding="utf-8") as f:
        for piece_id in range(sp.get_piece_size()):
            f.write(sp.id_to_piece(piece_id) + "\n")

    samples = [
        "tôi đang học tiếng việt",
        "vườn thuốc nam của trường có đủ các loại cây trồng trên luống",
        "đây là một câu kiểm tra đầy đủ dấu tiếng việt",
    ]
    print("Tokenizer dir:", TOKENIZER_DIR)
    print("Tokenizer vocab size:", sp.get_piece_size())
    print("Tokenizer vocab.txt:", vocab_txt)
    for text in samples:
        ids = sp.encode(text)
        decoded = sp.decode(ids)
        print("TOKENIZER CHECK:", text, "=>", decoded)
        assert normalize_for_metric(decoded) == normalize_for_metric(text)


## 5. NeMo Data and Model Setup Helpers


In [21]:
def dataloader_cfg(manifest_path: Path, batch_size: int, shuffle: bool, max_duration: float | None = None):
    cfg = {
        "manifest_filepath": str(manifest_path),
        "sample_rate": 16000,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": 2,
        "pin_memory": True,
        "min_duration": MIN_DURATION,
        "is_tarred": False,
        "tarred_audio_filepaths": None,
    }
    if max_duration is not None:
        cfg["max_duration"] = max_duration
    return OmegaConf.create(cfg)


def setup_validation_data(model, cfg) -> None:
    try:
        model.setup_multiple_validation_data(cfg)
    except Exception:
        model.setup_validation_data(cfg)


def load_phase_model(init_nemo: Path | None):
    if init_nemo is None:
        print("Loading pretrained English checkpoint:", MODEL_NAME)
        model = nemo_asr.models.EncDecCTCModelBPE.from_pretrained(model_name=MODEL_NAME)
        model.change_vocabulary(new_tokenizer_dir=str(TOKENIZER_DIR), new_tokenizer_type="bpe")
        return model

    if not init_nemo.exists():
        raise FileNotFoundError(f"Missing phase init model: {init_nemo}")
    print("Restoring phase init model:", init_nemo)
    return nemo_asr.models.ASRModel.restore_from(str(init_nemo))


def encoder_blocks(model):
    blocks = getattr(model.encoder, "layers", None)
    if blocks is not None and len(blocks) > 0:
        return blocks, "encoder.layers"

    candidates = []
    for name, module in model.encoder.named_modules():
        if hasattr(module, "__len__") and hasattr(module, "__iter__"):
            try:
                length = len(module)
            except Exception:
                continue
            if length >= 2 and ("layer" in name.lower() or "block" in name.lower()):
                candidates.append((length, name, module))
    if not candidates:
        raise RuntimeError("Could not find encoder layers for partial unfreeze.")
    candidates.sort(reverse=True, key=lambda item: item[0])
    _, name, module = candidates[0]
    return module, name


def set_trainable_mode(model, mode: str, unfrozen_encoder_blocks: int = 0) -> dict[str, Any]:
    for parameter in model.parameters():
        parameter.requires_grad = False

    for parameter in model.decoder.parameters():
        parameter.requires_grad = True

    info: dict[str, Any] = {
        "trainable_mode": mode,
        "encoder_block_path": None,
        "unfrozen_encoder_blocks": 0,
    }

    if mode == "decoder_only":
        model.encoder.eval()
    elif mode == "partial_encoder_last":
        blocks, block_path = encoder_blocks(model)
        n_blocks = unfrozen_encoder_blocks or 6
        if len(blocks) < n_blocks:
            raise ValueError(f"Requested {n_blocks} blocks but encoder only has {len(blocks)}.")
        for block in list(blocks)[-n_blocks:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        model.encoder.train()
        info["encoder_block_path"] = block_path
        info["unfrozen_encoder_blocks"] = n_blocks
    elif mode == "full_encoder_decoder":
        for parameter in model.encoder.parameters():
            parameter.requires_grad = True
        model.encoder.train()
        info["encoder_block_path"] = "encoder"
        info["unfrozen_encoder_blocks"] = "all"
    else:
        raise ValueError(f"Unknown trainable mode: {mode}")

    encoder_trainable = sum(p.numel() for p in model.encoder.parameters() if p.requires_grad)
    decoder_trainable = sum(p.numel() for p in model.decoder.parameters() if p.requires_grad)
    total_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]

    info.update(
        {
            "encoder_trainable_params": encoder_trainable,
            "decoder_trainable_params": decoder_trainable,
            "total_trainable_params": total_trainable,
            "total_params": total_params,
            "trainable_fraction": total_trainable / max(total_params, 1),
        }
    )

    print("Trainable mode:", mode)
    print("Encoder trainable params:", f"{encoder_trainable:,}")
    print("Decoder trainable params:", f"{decoder_trainable:,}")
    print("Total trainable params:", f"{total_trainable:,}/{total_params:,}")
    print("First trainable parameter names:", trainable_names[:20])

    if mode == "decoder_only":
        assert encoder_trainable == 0, "Encoder should be frozen in decoder-only mode."
    else:
        assert encoder_trainable > 0, "Encoder should be trainable in this phase."
    assert decoder_trainable > 0, "Decoder should always be trainable."
    return info


## 6. Training Callbacks


In [22]:
class EncoderModeCallback(Callback):
    def __init__(self, mode: str):
        self.mode = mode

    def _apply(self, pl_module):
        if self.mode == "decoder_only":
            pl_module.encoder.eval()
        else:
            pl_module.encoder.train()

    def on_train_start(self, trainer, pl_module):
        self._apply(pl_module)

    def on_train_epoch_start(self, trainer, pl_module):
        self._apply(pl_module)

    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        self._apply(pl_module)


def metric_to_float(value):
    if value is None:
        return None
    if hasattr(value, "detach"):
        value = value.detach().cpu()
    try:
        return float(value)
    except Exception:
        return None


def metric_lookup(metrics: dict, *names: str):
    for name in names:
        if name in metrics:
            return metric_to_float(metrics[name])
    for key, value in metrics.items():
        key_str = str(key)
        if any(key_str == name or key_str.endswith(name) for name in names):
            return metric_to_float(value)
    return None


class BestNemoExporter(Callback):
    def __init__(self, output_path: Path, monitor: str = "val_wer", mode: str = "min"):
        self.output_path = Path(output_path)
        self.monitor = monitor
        self.mode = mode
        self.best_score = math.inf if mode == "min" else -math.inf
        self.best_epoch = None

    def _is_better(self, score: float) -> bool:
        return score < self.best_score if self.mode == "min" else score > self.best_score

    def on_validation_epoch_end(self, trainer, pl_module):
        if getattr(trainer, "sanity_checking", False):
            return
        score = metric_lookup(getattr(trainer, "callback_metrics", {}) or {}, self.monitor)
        if score is None or not self._is_better(score):
            return
        self.best_score = score
        self.best_epoch = trainer.current_epoch
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        pl_module.save_to(str(self.output_path))
        print(
            f"[best {self.monitor}] epoch={trainer.current_epoch + 1} "
            f"{self.monitor}={score:.6f} exported={self.output_path}"
        )


class EpochPredictionLogger(Callback):
    def __init__(self, manifest_path: Path, output_csv: Path, num_samples: int = 10, batch_size: int = 4):
        self.output_csv = Path(output_csv)
        self.batch_size = batch_size
        self.rows = []
        self.samples = []
        with open(manifest_path, encoding="utf-8") as f:
            for idx, line in enumerate(f):
                if idx >= num_samples:
                    break
                row = json.loads(line)
                self.samples.append(row)

    def on_validation_epoch_end(self, trainer, pl_module):
        if getattr(trainer, "sanity_checking", False) or not self.samples:
            return

        audio_paths = [row["audio_filepath"] for row in self.samples]
        refs = [row["text"] for row in self.samples]
        try:
            predictions = pl_module.transcribe(audio_paths, batch_size=self.batch_size, verbose=False)
        except TypeError:
            predictions = pl_module.transcribe(audio_paths, batch_size=self.batch_size)
        except Exception as error:
            print("Sample prediction logging failed:", repr(error))
            return

        clean_predictions = []
        for pred in predictions:
            clean_predictions.append(getattr(pred, "text", pred))

        print(f"\nSample predictions after epoch {trainer.current_epoch + 1}:")
        for idx, (audio_path, ref, hyp) in enumerate(zip(audio_paths, refs, clean_predictions)):
            hyp = normalize_for_metric(str(hyp))
            ref_norm = normalize_for_metric(ref)
            print(f"  [{idx}] REF: {ref_norm}")
            print(f"      HYP: {hyp}")
            self.rows.append(
                {
                    "epoch": trainer.current_epoch,
                    "global_step": trainer.global_step,
                    "idx": idx,
                    "audio_filepath": audio_path,
                    "ref": ref_norm,
                    "hyp": hyp,
                }
            )

        self.output_csv.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(self.rows).to_csv(self.output_csv, index=False, encoding="utf-8-sig")


class EpochMetricsLogger(Callback):
    def __init__(self):
        self.rows = []

    def on_validation_epoch_end(self, trainer, pl_module):
        if getattr(trainer, "sanity_checking", False):
            return
        metrics = getattr(trainer, "callback_metrics", {}) or {}
        row = {
            "epoch": trainer.current_epoch,
            "global_step": trainer.global_step,
            "val_loss": metric_lookup(metrics, "val_loss"),
            "val_wer": metric_lookup(metrics, "val_wer"),
        }
        if torch.cuda.is_available():
            row.update(
                {
                    "gpu_allocated_mb": torch.cuda.memory_allocated() / (1024 ** 2),
                    "gpu_reserved_mb": torch.cuda.memory_reserved() / (1024 ** 2),
                    "gpu_peak_allocated_mb": torch.cuda.max_memory_allocated() / (1024 ** 2),
                }
            )
        self.rows.append(row)
        print(
            "[epoch]"
            f" epoch={trainer.current_epoch + 1}/{trainer.max_epochs}"
            f" step={trainer.global_step}"
            f" val_loss={row['val_loss']}"
            f" val_wer={row['val_wer']}"
        )


## 7. Checkpointing, Optimizer, and Phase Runner


In [23]:
def create_checkpoint_callback(phase: PhaseConfig) -> ModelCheckpoint:
    checkpoint_dir = PHASE_ROOT / phase.name / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    kwargs = {
        "dirpath": str(checkpoint_dir),
        "filename": "best-{epoch:02d}-{val_wer:.4f}",
        "monitor": "val_wer",
        "mode": "min",
        "save_top_k": 1,
        "save_last": True,
        "auto_insert_metric_name": False,
    }
    try:
        return ModelCheckpoint(every_n_epochs=1, **kwargs)
    except TypeError:
        return ModelCheckpoint(**kwargs)


def optimizer_cfg(lr: float):
    return OmegaConf.create(
        {
            "name": "adamw",
            "lr": lr,
            "betas": [0.9, 0.98],
            "weight_decay": WEIGHT_DECAY,
            "sched": {
                "name": "CosineAnnealing",
                "warmup_steps": WARMUP_STEPS,
                "min_lr": lr / 10,
            },
        }
    )


def count_manifest_rows(manifest_path: Path, max_duration: float | None = None) -> int:
    count = 0
    with open(manifest_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            duration = float(row.get("duration", 0.0))
            if duration < MIN_DURATION:
                continue
            if max_duration is not None and duration > max_duration:
                continue
            count += 1
    return count


def run_phase(phase: PhaseConfig) -> dict[str, Any]:
    print("\n" + "=" * 80)
    print("Starting", phase.name)
    print(asdict(phase))
    print("=" * 80)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    phase_dir = PHASE_ROOT / phase.name
    phase_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_callback = create_checkpoint_callback(phase)
    best_exporter = BestNemoExporter(phase.output_nemo, monitor="val_wer", mode="min")
    sample_logger = EpochPredictionLogger(
        VAL_MANIFEST,
        ARTIFACT_DIR / f"{phase.name}_sample_predictions.csv",
        num_samples=NUM_SAMPLE_PREDICTIONS_PER_EPOCH,
    )
    metric_logger = EpochMetricsLogger()
    early_stopping = EarlyStopping(
        monitor="val_wer",
        mode="min",
        patience=EARLY_STOPPING_PATIENCE,
        min_delta=EARLY_STOPPING_MIN_DELTA,
    )

    trainer = pl.Trainer(
        devices=1,
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        strategy="auto",
        precision=PRECISION,
        max_epochs=phase.max_epochs,
        accumulate_grad_batches=phase.accumulate_grad_batches,
        gradient_clip_val=GRADIENT_CLIP_VAL,
        log_every_n_steps=50,
        enable_checkpointing=True,
        enable_progress_bar=True,
        enable_model_summary=False,
        logger=False,
        callbacks=[
            checkpoint_callback,
            best_exporter,
            sample_logger,
            metric_logger,
            early_stopping,
            EncoderModeCallback(phase.trainable_mode),
        ],
        num_sanity_val_steps=2,
        default_root_dir=str(phase_dir),
    )

    model = load_phase_model(phase.init_nemo)
    model.set_trainer(trainer)
    model.setup_training_data(
        dataloader_cfg(TRAIN_MANIFEST, phase.batch_size, shuffle=True, max_duration=phase.max_duration)
    )
    setup_validation_data(model, dataloader_cfg(VAL_MANIFEST, phase.batch_size, shuffle=False))
    trainable_info = set_trainable_mode(
        model,
        phase.trainable_mode,
        unfrozen_encoder_blocks=phase.unfrozen_encoder_blocks,
    )
    model.setup_optimization(optimizer_cfg(phase.lr))

    train_samples = count_manifest_rows(TRAIN_MANIFEST, max_duration=phase.max_duration)
    steps_per_epoch = math.ceil(train_samples / (phase.batch_size * phase.accumulate_grad_batches))
    print("effective_batch_size:", phase.batch_size * phase.accumulate_grad_batches)
    print("steps_per_epoch:", steps_per_epoch)
    print("estimated_total_steps:", steps_per_epoch * phase.max_epochs)
    print("checkpoint_dir:", checkpoint_callback.dirpath)
    print("best_nemo_target:", phase.output_nemo)

    ckpt_path = None
    configured_resume = RESUME_CKPT_BY_PHASE.get(phase.name, "")
    if RESUME_FROM_LAST and configured_resume:
        ckpt_path = str(Path(configured_resume))
        print("Resuming trainer state from:", ckpt_path)

    start_time = time.perf_counter()
    trainer.fit(model, ckpt_path=ckpt_path)
    elapsed_minutes = (time.perf_counter() - start_time) / 60

    if not phase.output_nemo.exists():
        print("No best .nemo was exported; saving current model as fallback:", phase.output_nemo)
        model.save_to(str(phase.output_nemo))

    phase_metrics_csv = ARTIFACT_DIR / f"{phase.name}_epoch_metrics.csv"
    if metric_logger.rows:
        df = pd.DataFrame(metric_logger.rows)
        df.insert(0, "phase", phase.name)
        df.to_csv(phase_metrics_csv, index=False, encoding="utf-8-sig")

    summary = {
        **asdict(phase),
        **trainable_info,
        "best_val_wer": best_exporter.best_score if math.isfinite(best_exporter.best_score) else None,
        "best_epoch": best_exporter.best_epoch,
        "best_nemo_path": str(phase.output_nemo),
        "best_checkpoint_path": checkpoint_callback.best_model_path,
        "last_checkpoint_path": str(Path(checkpoint_callback.dirpath) / "last.ckpt"),
        "elapsed_minutes": elapsed_minutes,
        "steps_per_epoch": steps_per_epoch,
        "estimated_total_steps": steps_per_epoch * phase.max_epochs,
        "precision": PRECISION,
        "warmup_steps": WARMUP_STEPS,
        "weight_decay": WEIGHT_DECAY,
    }

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


## 8. Evaluation and Artifact Helpers


In [24]:
def read_manifest_for_eval(manifest_path: Path) -> tuple[list[str], list[str]]:
    audio_paths = []
    refs = []
    with open(manifest_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            audio_paths.append(row["audio_filepath"])
            refs.append(row["text"])
    return audio_paths, refs


def transcribe_model(model, audio_paths: list[str], batch_size: int) -> list[str]:
    predictions = []
    for start in range(0, len(audio_paths), batch_size):
        batch = audio_paths[start : start + batch_size]
        try:
            batch_preds = model.transcribe(batch, batch_size=batch_size, verbose=False)
        except TypeError:
            batch_preds = model.transcribe(batch, batch_size=batch_size)
        predictions.extend(getattr(pred, "text", pred) for pred in batch_preds)
    return [normalize_for_metric(str(pred)) for pred in predictions]


def evaluate_nemo(nemo_path: Path, manifest_path: Path, split: str, phase_name: str) -> dict[str, Any]:
    print(f"Evaluating {phase_name} on {split}: {nemo_path}")
    audio_paths, refs = read_manifest_for_eval(manifest_path)
    refs_norm = [normalize_for_metric(ref) for ref in refs]

    model = nemo_asr.models.ASRModel.restore_from(str(nemo_path))
    if torch.cuda.is_available():
        model = model.cuda()
    model.eval()

    start_time = time.perf_counter()
    preds_norm = transcribe_model(model, audio_paths, batch_size=EVAL_BATCH_SIZE)
    inference_seconds = time.perf_counter() - start_time

    row = {
        "phase": phase_name,
        "split": split,
        "model_path": str(nemo_path),
        "samples": len(refs_norm),
        "wer": wer(refs_norm, preds_norm),
        "cer": cer(refs_norm, preds_norm),
        "empty_pred_rate": sum(1 for pred in preds_norm if not pred) / max(len(preds_norm), 1),
        "ref_word_count": sum(len(ref.split()) for ref in refs_norm),
        "pred_word_count": sum(len(pred.split()) for pred in preds_norm),
        "inference_seconds": inference_seconds,
    }
    row["pred_ref_word_ratio"] = row["pred_word_count"] / max(row["ref_word_count"], 1)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(row)
    return row


def select_overall_best(eval_rows: list[dict[str, Any]]) -> None:
    val_rows = [row for row in eval_rows if row["split"] == "val"]
    if not val_rows:
        print("No validation eval rows; skipping overall best selection.")
        return
    best_row = sorted(val_rows, key=lambda row: row["wer"])[0]
    source = Path(best_row["model_path"])
    shutil.copy2(source, OVERALL_BEST_NEMO)
    print("Overall best by val WER:", best_row)
    print("Copied overall best to:", OVERALL_BEST_NEMO)


def combine_sample_predictions() -> None:
    frames = []
    for phase in PHASES:
        path = ARTIFACT_DIR / f"{phase.name}_sample_predictions.csv"
        if path.exists():
            frame = pd.read_csv(path)
            frame.insert(0, "phase", phase.name)
            frames.append(frame)
    if frames:
        pd.concat(frames, ignore_index=True).to_csv(SAMPLE_PREDICTIONS_CSV, index=False, encoding="utf-8-sig")


def csv_safe_summary(summary: dict[str, Any]) -> dict[str, Any]:
    safe = {}
    for key, value in summary.items():
        if isinstance(value, Path):
            safe[key] = str(value)
        else:
            safe[key] = value
    return safe


def run_phase_and_eval(
    phase: PhaseConfig,
    training_rows: list[dict[str, Any]],
    eval_rows: list[dict[str, Any]],
) -> dict[str, Any]:
    summary = run_phase(phase)
    training_rows.append(csv_safe_summary(summary))
    pd.DataFrame(training_rows).to_csv(TRAINING_CSV, index=False, encoding="utf-8-sig")

    eval_rows.append(evaluate_nemo(phase.output_nemo, VAL_MANIFEST, "val", phase.name))
    eval_rows.append(evaluate_nemo(phase.output_nemo, TEST_MANIFEST, "test", phase.name))
    pd.DataFrame(eval_rows).to_csv(EVAL_CSV, index=False, encoding="utf-8-sig")
    combine_sample_predictions()
    return summary


## 9. Optional Full Pipeline Function

Hàm `main()` vẫn được giữ để chạy tự động toàn bộ nếu muốn, nhưng workflow chính bên dưới chạy từng phase riêng.


In [25]:
def main() -> None:
    for directory in [MANIFEST_DIR, TOKENIZER_ROOT, EXP_DIR, ARTIFACT_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))

    prepare_manifests()
    train_sentencepiece_tokenizer()

    training_rows = []
    eval_rows = []
    for phase in PHASES:
        run_phase_and_eval(phase, training_rows, eval_rows)

    select_overall_best(eval_rows)

    print("\nArtifacts:")
    for path in [
        PHASE1_BEST_NEMO,
        PHASE2_BEST_NEMO,
        PHASE3_BEST_NEMO,
        OVERALL_BEST_NEMO,
        TRAINING_CSV,
        EVAL_CSV,
        SAMPLE_PREDICTIONS_CSV,
    ]:
        if path.exists():
            print(path, f"{path.stat().st_size / (1024 ** 2):.2f} MB")
        else:
            print("Missing:", path)
    print("\nCheckpoint folders:")
    for phase in PHASES:
        print(phase.name, PHASE_ROOT / phase.name / "checkpoints")


## 10. Prepare Data and Tokenizer

Chạy cell này trước các phase. Cell này tạo manifest, train tokenizer, và sinh `vocab.txt` để NeMo `change_vocabulary()` không lỗi.


In [26]:
for directory in [MANIFEST_DIR, TOKENIZER_ROOT, EXP_DIR, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

prepare_manifests()
train_sentencepiece_tokenizer()

training_rows = []
eval_rows = []
print("Ready for Phase 1")


torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
   split   rows    sample_rates  min_duration  max_duration  mean_duration  \
0  train  11077  {16000: 11077}      0.811875     17.125000       4.599300   
1    val    583    {16000: 583}      1.187500     17.000000       4.750377   
2   test    760    {16000: 760}      1.375000      7.843125       3.533611   

   filtered_by_max_duration_8s  
0                         1030  
1                           70  
2                            0  
Tokenizer dir: /kaggle/working/tokenizers/tokenizer_spe_unigram_v1024_vi
Tokenizer vocab size: 1024
Tokenizer vocab.txt: /kaggle/working/tokenizers/tokenizer_spe_unigram_v1024_vi/vocab.txt
TOKENIZER CHECK: tôi đang học tiếng việt => tôi đang học tiếng việt
TOKENIZER CHECK: vườn thuốc nam của trường có đủ các loại cây trồng trên luống => vườn thuốc nam của trường có đủ các loại cây trồng trên luống
TOKENIZER CHECK: đây là một câu kiểm tra đầy đủ dấu tiếng việt => đây là một câu kiểm tra đầy đủ dấ

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: /kaggle/working/tokenizers/vivos_train_only_corpus_vi.txt
  input_format: 
  model_prefix: /kaggle/working/tokenizers/tokenizer_spe_unigram_v1024_vi/tokenizer
  model_type: UNIGRAM
  vocab_size: 1024
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 1000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 0
  use_all_vocab: 0
  unk_id: 0
  bos_id: -1
  eos_id: -1
  pad_id: -1
  unk_piece: <unk>
  bos_

## 11. Phase 1 - Decoder Warmup

Freeze encoder, train Vietnamese CTC decoder only, export `/kaggle/working/phase1_best_decoder_only.nemo`.


In [27]:
# phase1_summary = run_phase_and_eval(PHASES[0], training_rows, eval_rows)
# phase1_summary


## 12. Phase 2 - Partial Encoder Fine-Tune

Load Phase 1 best model, unfreeze the last 6 encoder blocks plus decoder, export `/kaggle/working/phase2_best_partial_encoder.nemo`.


In [28]:
# phase2_summary = run_phase_and_eval(PHASES[1], training_rows, eval_rows)
# phase2_summary


## 13. Phase 3 - Full Fine-Tune

Load Phase 2 best model, unfreeze the full encoder plus decoder, export `/kaggle/working/phase3_best_full_finetune.nemo`.


In [29]:
PHASE3_RESUME_NEMO = Path("/kaggle/input/models/trnmnhtn/finetune-fastconformer/other/default/1/phase3_best_full_finetune.nemo")

phase3_continue = PhaseConfig(
    name="phase3_continue_from_epoch11",
    output_nemo=Path("/kaggle/working/phase3_continue_best.nemo"),
    init_nemo=PHASE3_RESUME_NEMO,
    trainable_mode="full_encoder_decoder",
    max_epochs=12,
    lr=2e-5,
    batch_size=2,
    accumulate_grad_batches=16,
)

In [31]:
phase3_summary = run_phase_and_eval(phase3_continue, training_rows, eval_rows)
phase3_summary


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



Starting phase3_continue_from_epoch11
{'name': 'phase3_continue_from_epoch11', 'output_nemo': PosixPath('/kaggle/working/phase3_continue_best.nemo'), 'init_nemo': PosixPath('/kaggle/input/models/trnmnhtn/finetune-fastconformer/other/default/1/phase3_best_full_finetune.nemo'), 'trainable_mode': 'full_encoder_decoder', 'max_epochs': 12, 'lr': 2e-05, 'batch_size': 2, 'accumulate_grad_batches': 16, 'unfrozen_encoder_blocks': 0, 'max_duration': 8.0}
Restoring phase init model: /kaggle/input/models/trnmnhtn/finetune-fastconformer/other/default/1/phase3_best_full_finetune.nemo
[NeMo I 2026-07-01 10:43:14 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-07-01 10:43:14 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/vivos_manifest_phase_best/train_manifest.json
    sample_rate: 16000
    batch_size: 2
    shuffle: true
    num_workers: 2
    pin_memory: true
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    max_duration: 8.0
    
[NeMo W 2026-07-01 10:43:14 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/vivos_manifest_phase_best/val_manifest.json
    sample_rate: 16000
    batch_size: 2
    shuffle: false
    num_workers: 2
    pin_memory: true
    min_duration: 0.1
 

[NeMo I 2026-07-01 10:43:17 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /kaggle/input/models/trnmnhtn/finetune-fastconformer/other/default/1/phase3_best_full_finetune.nemo.
[NeMo I 2026-07-01 10:43:17 collections:201] Dataset loaded with 10047 files totalling 11.29 hours
[NeMo I 2026-07-01 10:43:17 collections:202] 1030 files were filtered totalling 2.86 hours
[NeMo I 2026-07-01 10:43:17 collections:201] Dataset loaded with 583 files totalling 0.77 hours
[NeMo I 2026-07-01 10:43:17 collections:202] 0 files were filtered totalling 0.00 hours
Trainable mode: full_encoder_decoder
Encoder trainable params: 115,074,560
Decoder trainable params: 525,825
Total trainable params: 115,600,385/115,600,385
First trainable parameter names: ['encoder.pre_encode.out.weight', 'encoder.pre_encode.out.bias', 'encoder.pre_encode.conv.0.weight', 'encoder.pre_encode.conv.0.bias', 'encoder.pre_encode.conv.2.weight', 'encoder.pre_encode.conv.2.bias', 'encoder.pre_encode

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...

[NeMo W 2026-07-01 13:45:11 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/vivos_manifest_phase_best/train_manifest.json
    sample_rate: 16000
    batch_size: 2
    shuffle: true
    num_workers: 2
    pin_memory: true
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    max_duration: 8.0
    
[NeMo W 2026-07-01 13:45:11 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/vivos_manifest_phase_best/val_manifest.json
    sample_rate: 16000
    batch_size: 2
    shuffle: false
    num_workers: 2
    pin_memory: true
    min_duration: 0.1
 

[NeMo I 2026-07-01 13:45:13 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /kaggle/working/phase3_continue_best.nemo.


[NeMo W 2026-07-01 13:45:13 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-07-01 13:45:13 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
[NeMo W 2026-07-01 13:45:13 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-07-01 13:45:13 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is

{'phase': 'phase3_continue_from_epoch11', 'split': 'test', 'model_path': '/kaggle/working/phase3_continue_best.nemo', 'samples': 760, 'wer': 0.4783734783734784, 'cer': 0.22471087241768636, 'empty_pred_rate': 0.0, 'ref_word_count': 7722, 'pred_word_count': 7723, 'inference_seconds': 9.637337202999333, 'pred_ref_word_ratio': 1.0001295001295}


{'name': 'phase3_continue_from_epoch11',
 'output_nemo': PosixPath('/kaggle/working/phase3_continue_best.nemo'),
 'init_nemo': PosixPath('/kaggle/input/models/trnmnhtn/finetune-fastconformer/other/default/1/phase3_best_full_finetune.nemo'),
 'trainable_mode': 'full_encoder_decoder',
 'max_epochs': 12,
 'lr': 2e-05,
 'batch_size': 2,
 'accumulate_grad_batches': 16,
 'unfrozen_encoder_blocks': 'all',
 'max_duration': 8.0,
 'encoder_block_path': 'encoder',
 'encoder_trainable_params': 115074560,
 'decoder_trainable_params': 525825,
 'total_trainable_params': 115600385,
 'total_params': 115600385,
 'trainable_fraction': 1.0,
 'best_val_wer': 0.43843308091163635,
 'best_epoch': 11,
 'best_nemo_path': '/kaggle/working/phase3_continue_best.nemo',
 'best_checkpoint_path': '/kaggle/working/nemo_experiments/fastconformer_vi/phase3_continue_from_epoch11/checkpoints/best-11-0.4381.ckpt',
 'last_checkpoint_path': '/kaggle/working/nemo_experiments/fastconformer_vi/phase3_continue_from_epoch11/checkp

In [33]:
import pandas as pd
from jiwer import wer, cer

rows = []

model_path = "/kaggle/working/phase3_continue_best.nemo"
model = nemo_asr.models.ASRModel.restore_from(model_path)
model.eval()

audio_paths, refs = read_manifest_for_eval(TEST_MANIFEST)
hyps = model.transcribe(audio_paths, batch_size=8, verbose=False)

for audio, ref, hyp in zip(audio_paths, refs, hyps):
    ref_n = normalize_for_metric(ref)
    hyp_n = normalize_for_metric(hyp)
    rows.append({
        "audio": audio,
        "ref": ref_n,
        "hyp": hyp_n,
        "wer": wer(ref_n, hyp_n),
        "cer": cer(ref_n, hyp_n),
        "ref_len_words": len(ref_n.split()),
        "hyp_len_words": len(hyp_n.split()),
        "ref_len_chars": len(ref_n),
        "hyp_len_chars": len(hyp_n),
    })

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/error_analysis_test.csv", index=False, encoding="utf-8-sig")
df.head()

[NeMo I 2026-07-01 13:51:06 mixins:184] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-07-01 13:51:06 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /kaggle/working/vivos_manifest_phase_best/train_manifest.json
    sample_rate: 16000
    batch_size: 2
    shuffle: true
    num_workers: 2
    pin_memory: true
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    max_duration: 8.0
    
[NeMo W 2026-07-01 13:51:06 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation config : 
    manifest_filepath: /kaggle/working/vivos_manifest_phase_best/val_manifest.json
    sample_rate: 16000
    batch_size: 2
    shuffle: false
    num_workers: 2
    pin_memory: true
    min_duration: 0.1
 

[NeMo I 2026-07-01 13:51:08 save_restore_connector:285] Model EncDecCTCModelBPE was successfully restored from /kaggle/working/phase3_continue_best.nemo.


[NeMo W 2026-07-01 13:51:08 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-07-01 13:51:08 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)


,audio,ref,hyp,wer,cer,ref_len_words,hyp_len_words,ref_len_chars,hyp_len_chars
0,/kaggle/working/vivos/data/extracted/vivos/tes...,trở nên thụ động,hypothesis score tensor 3 5975 y sequence tens...,23.500000,31.187500,4,95,16,512
1,/kaggle/working/vivos/data/extracted/vivos/tes...,cũng khiến cho họ dè dặt,hypothesis score tensor 6 0061 y sequence tens...,15.500000,20.333333,6,96,24,506
2,/kaggle/working/vivos/data/extracted/vivos/tes...,chị gặn hỏi anh thề sống thề chết là không có,hypothesis score tensor 7 0633 y sequence tens...,10.545455,13.222222,11,123,45,634
3,/kaggle/working/vivos/data/extracted/vivos/tes...,điện thoại reng nhưng ta cũng phải nhấc máy đú...,hypothesis score tensor 9 8388 y sequence tens...,10.181818,10.592593,11,118,54,614
4,/kaggle/working/vivos/data/extracted/vivos/tes...,cũng thuộc loại dày đặc nhất trong hệ mặt trời,hypothesis score tensor 10 2570 y sequence ten...,11.000000,12.217391,10,115,46,595


In [35]:
df.sort_values("wer", ascending=False).head(30)[
    ["wer", "cer", "ref_len_words", "hyp_len_words", "ref", "hyp", "audio"]
]

,wer,cer,ref_len_words,hyp_len_words,ref,hyp,audio
167,43.500000,58.500000,2,87,trán giô,hypothesis score tensor 5 5906 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
222,40.000000,48.444444,2,81,làm khung,hypothesis score tensor 1 0211 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
58,39.000000,70.833333,2,78,ôi dào,hypothesis score tensor 1 8092 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
120,38.500000,70.666667,2,78,bà nói,hypothesis score tensor 1 2710 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
445,32.666667,51.900000,3,98,tại chợ cá,hypothesis score tensor 2 5723 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
141,30.333333,39.916667,3,91,rạch bàu mớp,hypothesis score tensor 5 8160 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
315,29.333333,43.909091,3,91,quá tốt rồi,hypothesis score tensor 1 8762 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
139,28.666667,51.222222,3,87,vì tối đó,hypothesis score tensor 1 9623 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
428,28.666667,39.250000,3,89,năm đầu tiên,hypothesis score tensor 1 8397 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...
100,27.666667,38.000000,3,86,người ta nói,hypothesis score tensor 0 9303 y sequence tens...,/kaggle/working/vivos/data/extracted/vivos/tes...


In [36]:
df["len_bucket"] = pd.cut(
    df["ref_len_words"],
    bins=[0, 3, 6, 10, 15, 25, 100],
    labels=["1-3", "4-6", "7-10", "11-15", "16-25", "25+"]
)

df.groupby("len_bucket")[["wer", "cer"]].mean()

,wer,cer
len_bucket,,
1-3,31.571429,46.384323
4-6,17.225969,22.152369
7-10,12.112919,14.929639
11-15,9.195368,10.919638
16-25,7.900463,9.670945
25+,NaN,NaN


In [37]:
df["word_len_ratio"] = df["hyp_len_words"] / df["ref_len_words"].clip(lower=1)

df[["word_len_ratio", "wer", "cer"]].describe()

,word_len_ratio,wer,cer
count,760.000000,760.000000,760.000000
mean,12.273153,11.733312,14.529878
std,4.319054,4.321519,6.618598
min,7.714286,7.142857,7.469697
25%,9.615385,9.076923,10.613636
50%,10.727273,10.230769,12.364525
75%,13.647321,13.129464,16.359524
max,43.500000,43.500000,70.833333


In [38]:
df[df["word_len_ratio"] < 0.7].sort_values("wer", ascending=False).head(20)[
    ["word_len_ratio", "wer", "ref", "hyp"]
]

,word_len_ratio,wer,ref,hyp


In [39]:
df[df["word_len_ratio"] > 1.3].sort_values("wer", ascending=False).head(20)[
    ["word_len_ratio", "wer", "ref", "hyp"]
]

,word_len_ratio,wer,ref,hyp
167,43.500000,43.500000,trán giô,hypothesis score tensor 5 5906 y sequence tens...
222,40.500000,40.000000,làm khung,hypothesis score tensor 1 0211 y sequence tens...
58,39.000000,39.000000,ôi dào,hypothesis score tensor 1 8092 y sequence tens...
120,39.000000,38.500000,bà nói,hypothesis score tensor 1 2710 y sequence tens...
445,32.666667,32.666667,tại chợ cá,hypothesis score tensor 2 5723 y sequence tens...
141,30.333333,30.333333,rạch bàu mớp,hypothesis score tensor 5 8160 y sequence tens...
315,30.333333,29.333333,quá tốt rồi,hypothesis score tensor 1 8762 y sequence tens...
139,29.000000,28.666667,vì tối đó,hypothesis score tensor 1 9623 y sequence tens...
428,29.666667,28.666667,năm đầu tiên,hypothesis score tensor 1 8397 y sequence tens...
100,28.666667,27.666667,người ta nói,hypothesis score tensor 0 9303 y sequence tens...


In [41]:
import unicodedata

def remove_accents(text):
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", text).replace("đ", "d")

df["wer_no_accent"] = [
    wer(remove_accents(r), remove_accents(h))
    for r, h in zip(df["ref"], df["hyp"])
]

df["accent_penalty"] = df["wer"] - df["wer_no_accent"]

df.sort_values("accent_penalty", ascending=False).head(30)[
    ["wer", "wer_no_accent", "accent_penalty", "ref", "hyp"]
]

,wer,wer_no_accent,accent_penalty,ref,hyp
435,19.400000,18.600000,0.800000,trọng tài phạt thẻ đỏ,hypothesis score tensor 2 8885 y sequence tens...
423,23.750000,23.000000,0.750000,ngụ ấp hòn ngang,hypothesis score tensor 8 0874 y sequence tens...
194,12.000000,11.250000,0.750000,bọn buôn lậu ma túy và vũ khí,hypothesis score tensor 8 8167 y sequence tens...
457,14.428571,13.857143,0.571429,dưới ánh đèn mờ mờ ảo ảo,hypothesis score tensor 8 7676 y sequence tens...
349,16.500000,16.000000,0.500000,mười một mười hai mười ba,hypothesis score tensor 0 8638 y sequence tens...
0,23.500000,23.000000,0.500000,trở nên thụ động,hypothesis score tensor 3 5975 y sequence tens...
167,43.500000,43.000000,0.500000,trán giô,hypothesis score tensor 5 5906 y sequence tens...
120,38.500000,38.000000,0.500000,bà nói,hypothesis score tensor 1 2710 y sequence tens...
49,16.600000,16.200000,0.400000,mà cũng đầy thanh khiết,hypothesis score tensor 4 4251 y sequence tens...
168,17.600000,17.200000,0.400000,thu chạm ngõ rất hiền,hypothesis score tensor 6 6459 y sequence tens...


In [43]:
from collections import Counter
from difflib import SequenceMatcher

substitutions = Counter()
deletions = Counter()
insertions = Counter()

for ref, hyp in zip(df["ref"], df["hyp"]):
    r = ref.split()
    h = hyp.split()
    sm = SequenceMatcher(a=r, b=h)
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "replace":
            for a, b in zip(r[i1:i2], h[j1:j2]):
                substitutions[(a, b)] += 1
        elif tag == "delete":
            for a in r[i1:i2]:
                deletions[a] += 1
        elif tag == "insert":
            for b in h[j1:j2]:
                insertions[b] += 1

print("Substitution:")
print(substitutions.most_common(30))

print("Deletion:")
print(deletions.most_common(30))

print("Insertion:")
print(insertions.most_common(30))

Substitution:
[(('nay', 'này'), 9), (('vì', 'hypothesis'), 7), (('mạnh', 'mình'), 7), (('những', 'hypothesis'), 7), (('cùng', 'cũng'), 7), (('câu', 'cầu'), 7), (('nhưng', 'hypothesis'), 6), (('an', 'ăn'), 6), (('minh', 'mình'), 6), (('ảnh', 'anh'), 6), (('đáng', 'đang'), 6), (('sâu', 'sau'), 5), (('thanh', 'thành'), 5), (('gọi', 'gọ'), 5), (('đấy', 'đây'), 5), (('tự', 'từ'), 5), (('từng', 'từ'), 5), (('chị', 'chỉ'), 5), (('việt', 'việc'), 5), (('chị', 'hypothesis'), 4), (('lạ', 'là'), 4), (('tôi', 'hypothesis'), 4), (('hẳn', 'hàng'), 4), (('đề', 'để'), 4), (('trị', 'chỉ'), 4), (('băng', 'bằng'), 4), (('ngoại', 'ngoài'), 4), (('trọng', 'trong'), 4), (('bản', 'bạn'), 4), (('tội', 'tôi'), 4)]
Deletion:
[('thấy', 1), ('quá', 1), ('giảm', 1), ('ác', 1), ('với', 1), ('mình', 1), ('cực', 1), ('gan', 1), ('sò', 1), ('thịt', 1), ('đỏ', 1), ('chè', 1), ('nặng', 1), ('trĩu', 1), ('chợ', 1)]
Insertion:
[('1024', 15567), ('none', 6176), ('state', 1158), ('confidence', 1158), ('token', 1158), ('lm',

## 14. Finalize Best Model and Show Artifacts

Chọn overall best theo validation WER và in các file cần download.


In [ ]:
select_overall_best(eval_rows)

print("\nArtifacts:")
for path in [
    PHASE1_BEST_NEMO,
    PHASE2_BEST_NEMO,
    PHASE3_BEST_NEMO,
    OVERALL_BEST_NEMO,
    TRAINING_CSV,
    EVAL_CSV,
    SAMPLE_PREDICTIONS_CSV,
]:
    if path.exists():
        print(path, f"{path.stat().st_size / (1024 ** 2):.2f} MB")
    else:
        print("Missing:", path)

print("\nCheckpoint folders:")
for phase in PHASES:
    print(phase.name, PHASE_ROOT / phase.name / "checkpoints")

if eval_rows:
    display(pd.DataFrame(eval_rows))
if training_rows:
    display(pd.DataFrame(training_rows))
